# **07_models_lstm**

Este notebook implementa un modelo LSTM para la predicción de la demanda eléctrica diaria
a partir de la serie objetivo `target_kwh`.

## Objetivos
- Preparar la serie para modelado secuencial.
- Construir secuencias temporales mediante una ventana de observación.
- Entrenar un modelo LSTM univariante.
- Evaluar su desempeño sobre el conjunto de prueba.
- Generar tablas y figuras útiles para el TFM.

**BLOQUE 2 — Librerías y configuración**

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

**BLOQUE 3 — Rutas**

In [ ]:
# ==========================================
# BLOQUE 3 — Configuración de rutas
# ==========================================

from pathlib import Path

# Detectar si está dentro de notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUTS_FIGURES = PROJECT_ROOT / "outputs" / "figuras"
OUTPUTS_TABLES = PROJECT_ROOT / "outputs" / "tablas"

# Crear carpetas si no existen
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
OUTPUTS_FIGURES.mkdir(parents=True, exist_ok=True)
OUTPUTS_TABLES.mkdir(parents=True, exist_ok=True)

# Archivo de entrada
INPUT_FILE = DATA_PROCESSED / "data_daily_ready.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Archivo de entrada:", INPUT_FILE)

# Validación
if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo: {INPUT_FILE}\n"
        "Verifica que el archivo exista en data/processed/"
    )

**BLOQUE 5 — División train/test**

In [ ]:
TARGET_COL = "target_kwh"

df = pd.read_csv(INPUT_FILE, index_col="date", parse_dates=True)

train = df.loc["2022-01-01":"2024-12-31", [TARGET_COL]].copy()
test = df.loc["2025-01-01":"2025-12-10", [TARGET_COL]].copy()

print("Train shape:", train.shape)
print("Test shape :", test.shape)
print("Rango train:", train.index.min(), "->", train.index.max())
print("Rango test :", test.index.min(), "->", test.index.max())

**BLOQUE 4 — Carga de datos**

In [ ]:
df = pd.read_csv(INPUT_FILE, parse_dates=["date"])

DATE_COL = "date"
TARGET_COL = "target_kwh"

df = df.sort_values(DATE_COL).set_index(DATE_COL)

print("Dimensión del dataset:", df.shape)
print("Frecuencia inferida:", pd.infer_freq(df.index))

display(df[[TARGET_COL]].head())
display(df[[TARGET_COL]].tail())

**BLOQUE 5 — División train/test**

In [ ]:
train = df.loc["2022-01-01":"2024-12-31", [TARGET_COL]].copy()
test = df.loc["2025-01-01":"2025-12-10", [TARGET_COL]].copy()

print("Train shape:", train.shape)
print("Test shape :", test.shape)
print("Rango train:", train.index.min(), "->", train.index.max())
print("Rango test :", test.index.min(), "->", test.index.max())

**BLOQUE 6 — Funciones auxiliares**

In [ ]:
def mape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def compute_metrics(y_true, y_pred, model_name):
    return {
        "model": model_name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAPE": mape(y_true, y_pred)
    }

def create_sequences(data_array, lookback):
    X, y = [], []
    for i in range(lookback, len(data_array)):
        X.append(data_array[i-lookback:i, 0])
        y.append(data_array[i, 0])
    return np.array(X), np.array(y)

**BLOQUE 7 — Escalado**

In [ ]:
scaler = MinMaxScaler()

train_scaled = scaler.fit_transform(train)
test_scaled = scaler.transform(test)

print("Train scaled shape:", train_scaled.shape)
print("Test scaled shape :", test_scaled.shape)

**BLOQUE 8 — Creación de secuencias**

In [ ]:
LOOKBACK = 14

# Para test, concatenamos el final de train con test para construir secuencias válidas
full_scaled_for_test = np.vstack([train_scaled[-LOOKBACK:], test_scaled])

X_train, y_train = create_sequences(train_scaled, LOOKBACK)
X_test, y_test = create_sequences(full_scaled_for_test, LOOKBACK)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

**BLOQUE 9 — Reshape para LSTM**

In [ ]:
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

print("X_train reshaped:", X_train.shape)
print("X_test reshaped :", X_test.shape)

**BLOQUE 10 — Arquitectura del modelo**

In [ ]:
lstm_model = Sequential([
    LSTM(64, return_sequences=False, input_shape=(LOOKBACK, 1)),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(1)
])

lstm_model.compile(
    optimizer="adam",
    loss="mse"
)

lstm_model.summary()

**BLOQUE 10A — Función para construir modelos LSTM**

In [ ]:
# ==========================================
# BLOQUE 10A — Función para construir modelos LSTM
# ==========================================

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

def build_lstm_model(units=64, dense_units=32, dropout=0.2, stacked=False, input_shape=(14,1)):

    model = Sequential()

    if stacked:
        model.add(LSTM(units, return_sequences=True, input_shape=input_shape))
        model.add(Dropout(dropout))
        model.add(LSTM(units//2))
    else:
        model.add(LSTM(units, return_sequences=False, input_shape=input_shape))

    model.add(Dropout(dropout))
    model.add(Dense(dense_units, activation="relu"))
    model.add(Dense(1))

    model.compile(optimizer="adam", loss="mse")

    return model

**BLOQUE 10B — Función para crear secuencias con lookback variable**

In [ ]:
# ==========================================
# BLOQUE 10B — Función para crear secuencias dinámicas
# ==========================================

def create_sequences_dynamic(data_array, lookback):
    X, y = [], []
    for i in range(lookback, len(data_array)):
        X.append(data_array[i-lookback:i, 0])
        y.append(data_array[i, 0])
    return np.array(X), np.array(y)

**BLOQUE 10C — Definición de configuraciones a probar**

In [ ]:
# ==========================================
# BLOQUE 10C — Configuraciones LSTM a evaluar
# ==========================================

lstm_configs = [
    {"lookback": 14, "units": 64, "dense": 32, "stacked": False},
    {"lookback": 21, "units": 64, "dense": 32, "stacked": False},
    {"lookback": 28, "units": 64, "dense": 32, "stacked": False},
    {"lookback": 21, "units": 32, "dense": 16, "stacked": False},
    {"lookback": 28, "units": 64, "dense": 16, "stacked": True},
]

**BLOQUE 10D — Evaluación de configuraciones**

In [ ]:
# ==========================================
# BLOQUE 10D — Evaluación de configuraciones LSTM
# ==========================================

results_lstm = []

for config in lstm_configs:

    lookback = config["lookback"]

    # crear secuencias
    X_train_tmp, y_train_tmp = create_sequences_dynamic(train_scaled, lookback)

    full_scaled = np.vstack([train_scaled[-lookback:], test_scaled])
    X_test_tmp, y_test_tmp = create_sequences_dynamic(full_scaled, lookback)

    # reshape
    X_train_tmp = X_train_tmp.reshape((X_train_tmp.shape[0], X_train_tmp.shape[1], 1))
    X_test_tmp = X_test_tmp.reshape((X_test_tmp.shape[0], X_test_tmp.shape[1], 1))

    # modelo
    model = build_lstm_model(
        units=config["units"],
        dense_units=config["dense"],
        stacked=config["stacked"],
        input_shape=(lookback, 1)
    )

    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    )

    history = model.fit(
        X_train_tmp,
        y_train_tmp,
        validation_split=0.1,
        epochs=100,
        batch_size=32,
        callbacks=[early_stopping],
        verbose=0
    )

    # predicción
    y_pred_scaled = model.predict(X_test_tmp, verbose=0)

    y_pred = scaler.inverse_transform(y_pred_scaled).flatten()
    y_true = scaler.inverse_transform(y_test_tmp.reshape(-1,1)).flatten()

    metrics = compute_metrics(y_true, y_pred, "LSTM_test")

    results_lstm.append({
        "lookback": lookback,
        "units": config["units"],
        "dense": config["dense"],
        "stacked": config["stacked"],
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "MAPE": metrics["MAPE"]
    })

results_lstm_df = pd.DataFrame(results_lstm).sort_values("RMSE").reset_index(drop=True)

display(results_lstm_df)

**BLOQUE 10E — Selección del mejor modelo**

In [ ]:
# ==========================================
# BLOQUE 10E — Selección del mejor LSTM
# ==========================================

best_config = results_lstm_df.iloc[0]
print("Mejor configuración LSTM:")
print(best_config)

In [ ]:
# Guardar resultados de comparación LSTM
results_lstm_df.to_csv(OUTPUTS_TABLES / "table_lstm_config_comparison.csv", index=False)

**BLOQUE 11 — Entrenamiento**

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = lstm_model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

**BLOQUE 12 — Curvas de entrenamiento**

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(history.history["loss"], label="Train loss")
ax.plot(history.history["val_loss"], label="Validation loss")

ax.set_title("Curvas de entrenamiento - LSTM")
ax.set_xlabel("Época")
ax.set_ylabel("Loss (MSE)")
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.savefig(OUTPUTS_FIGURES / "lstm_training_curves.png", dpi=300, bbox_inches="tight")
plt.show()

**BLOQUE 13 — Predicción y desescalado**

In [ ]:
y_pred_scaled = lstm_model.predict(X_test)

y_pred = scaler.inverse_transform(y_pred_scaled).flatten()
y_true = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

# Índice temporal del test alineado con las predicciones
y_test_index = test.index

print("Predicciones:", y_pred.shape)
print("Valores reales:", y_true.shape)
print("Índice test:", len(y_test_index))

**BLOQUE 14 — Métricas**

In [ ]:
lstm_metrics = compute_metrics(y_true, y_pred, "LSTM")
display(pd.DataFrame([lstm_metrics]))

**BLOQUE 15 — Gráfico real vs predicción**

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4.8))

ax.plot(y_test_index, y_true, label="Real", linewidth=1.2)
ax.plot(y_test_index, y_pred, label="Predicción LSTM", linewidth=1.2)

ax.set_title("Predicción en test - LSTM", fontsize=13)
ax.set_xlabel("Fecha")
ax.set_ylabel("Demanda eléctrica diaria (kWh)")
ax.grid(True, alpha=0.3)
ax.legend(frameon=False)

ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

plt.tight_layout()
plt.savefig(OUTPUTS_FIGURES / "forecast_test_lstm.png", dpi=300, bbox_inches="tight")
plt.show()

**BLOQUE 16 — Exportación de predicciones**

In [ ]:
lstm_predictions_df = pd.DataFrame({
    "date": y_test_index,
    "y_true": y_true,
    "y_pred_lstm": y_pred
})

display(lstm_predictions_df.head())
lstm_predictions_df.to_csv(OUTPUTS_TABLES / "table_lstm_test_predictions.csv", index=False)

**BLOQUE 17 — Tabla de hiperparámetros**

In [ ]:
lstm_hyperparams = pd.DataFrame([{
    "model": "LSTM",
    "lookback": LOOKBACK,
    "lstm_units": 64,
    "dropout": 0.2,
    "dense_units": 32,
    "optimizer": "adam",
    "loss": "mse",
    "epochs_max": 100,
    "batch_size": 32,
    "validation_split": 0.1,
    "early_stopping_patience": 10
}])

display(lstm_hyperparams)
lstm_hyperparams.to_csv(OUTPUTS_TABLES / "table_hyperparameters_lstm.csv", index=False)

El modelo LSTM logró capturar la tendencia general de la serie temporal; sin embargo, presentó un desempeño inferior al de los modelos basados en árboles, especialmente XGBoost, con un MAPE significativamente mayor. Esto sugiere que, para este problema y tamaño de muestra, los modelos tabulares con variables derivadas y exógenas resultan más eficaces que los enfoques de deep learning univariantes.